# 02 — Preparação: filtros, variáveis derivadas e tabelas analíticas

**Objetivo:** aplicar o recorte da proposta e construir as tabelas analíticas com granularidade bem definida:

| Tabela | Granularidade | Uso |
|---|---|---|
| `pedidos` | 1 linha por `order_id` | KPI1, KPI3 geral, atraso de entrega |
| `itens` | 1 linha por `order_id` + `order_item_id` | KPI2 (frete/preço), categoria |
| `pedido_seller` | 1 linha por `order_id` + `seller_id` | ranking de sellers (**não** soma pedidos únicos) |
| `pedido_categoria` | 1 linha por `order_id` + categoria | KPIs por categoria |

### Premissa temporal (proxy acadêmica)
- **2024 (proxy)** = pedidos com compra em jan–ago/**2017** nos dados originais.
- **2025 (proxy)** = pedidos com compra em jan–ago/**2018** nos dados originais.

As datas originais **não são alteradas**. O dataset não contém vendas reais de 2024/2025; os rótulos só servem para a comparação equivalente pedida na proposta.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

import pandas as pd
import numpy as np
import olist_utils as U

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

descartes = []
def registrar(verificacao, valor, detalhe=""):
    descartes.append({"etapa": "02_preparacao", "tabela": "pipeline", "verificacao": verificacao,
                      "valor": valor, "detalhe": detalhe})

orders = U.load_raw("orders")
items = U.load_raw("items")
reviews = U.load_raw("reviews")
customers = U.load_raw("customers")
sellers = U.load_raw("sellers")
products = U.load_raw("products")
transl = U.load_raw("category_translation")

## 1. Filtro de status e período
Critério: `order_status == "delivered"` **e** ano de `order_purchase_timestamp` ∈ {2017, 2018} **e** mês ∈ {1..8}.

In [2]:
registrar("pedidos_base_total", len(orders))
o = orders.copy()
o["ano_original"] = o["order_purchase_timestamp"].dt.year
o["mes"] = o["order_purchase_timestamp"].dt.month

nao_entregue = o["order_status"] != "delivered"
fora_periodo = ~(o["ano_original"].isin(U.PERIODOS) & o["mes"].isin(U.MESES_RECORTE))
registrar("descarte_status_diferente_de_delivered", int(nao_entregue.sum()))
registrar("descarte_delivered_fora_do_periodo", int((~nao_entregue & fora_periodo).sum()),
          "delivered, mas fora de jan-ago/2017 e jan-ago/2018")

ped = o[~nao_entregue & ~fora_periodo].copy()
ped["periodo_comparativo"] = ped["ano_original"].map(U.PERIODOS)
ped["ano_mes_original"] = ped["order_purchase_timestamp"].dt.to_period("M").astype(str)

check = ped.groupby("ano_original").size().rename("calculado").to_frame()
check["referencia_proposta"] = check.index.map(U.CHECK_PEDIDOS_PROPOSTA)
check["diferenca"] = check["calculado"] - check["referencia_proposta"]
for ano, r in check.iterrows():
    registrar(f"pedidos_no_recorte_{ano}", int(r["calculado"]), f"referência da proposta: {r['referencia_proposta']}")
print("Total no recorte:", len(ped), "| referência da proposta: 74781")
check

Total no recorte: 74781 | referência da proposta: 74781


,calculado,referencia_proposta,diferenca
ano_original,,,
2017,21998,21998,0
2018,52783,52783,0


## 2. Cliente → UF e região
Junção 1:1 (`customer_id` é único em `customers`, verificado no notebook 01). A região vem de um mapa explícito das 27 UFs em `olist_utils.UF_REGIAO`.

In [3]:
n0 = len(ped)
ped = ped.merge(customers[["customer_id", "customer_unique_id", "customer_state", "customer_city"]],
                on="customer_id", how="left", validate="one_to_one")
assert len(ped) == n0
ped["regiao_cliente"] = ped["customer_state"].map(U.UF_REGIAO)
assert ped["regiao_cliente"].notna().all(), "UF sem região"
ped["regiao_cliente"].value_counts()

regiao_cliente
Sudeste         51436
Sul             10758
Nordeste         6879
Centro-Oeste     4321
Norte            1387
Name: count, dtype: int64

## 3. Avaliação: a mais recente por pedido
Regra: maior `review_creation_date`; desempate por `review_answer_timestamp` mais recente e depois maior `review_id` (determinístico). Pedidos sem avaliação **não** recebem nota imputada: ficam com `tem_avaliacao = False` e saem do denominador do KPI1.

In [4]:
rev_ult = U.latest_review_per_order(reviews)
assert rev_ult["order_id"].is_unique
multi = reviews[reviews["order_id"].isin(ped["order_id"])].groupby("order_id").size()
registrar("pedidos_recorte_com_multiplas_avaliacoes", int((multi > 1).sum()), "mantida a mais recente")

ped = ped.merge(rev_ult[["order_id", "review_id", "review_score", "review_creation_date"]],
                on="order_id", how="left", validate="one_to_one")
assert len(ped) == n0
ped["tem_avaliacao"] = ped["review_score"].notna()
ped["avaliacao_ruim"] = np.where(ped["tem_avaliacao"], ped["review_score"].isin([1, 2]), np.nan)
ped["avaliacao_ruim"] = ped["avaliacao_ruim"].astype("boolean")
registrar("pedidos_recorte_sem_avaliacao", int((~ped["tem_avaliacao"]).sum()), "fora do denominador do KPI1")
ped["tem_avaliacao"].value_counts()

tem_avaliacao
True     74305
False      476
Name: count, dtype: int64

## 4. Itens, produtos, categoria e sellers
- Categoria: usa a tradução para inglês; o nome original é preservado. Produtos sem categoria viram `sem_categoria`; categorias sem tradução mantêm o nome original em `categoria_en` e são sinalizadas.
- Os joins são todos N:1 (`validate="many_to_one"`), então **não** multiplicam itens nem frete.

In [5]:
it = items[items["order_id"].isin(ped["order_id"])].copy()
n_it = len(it)
it = it.merge(products[["product_id", "product_category_name"]], on="product_id", how="left", validate="many_to_one")
it = it.merge(transl, on="product_category_name", how="left", validate="many_to_one")
it = it.merge(sellers[["seller_id", "seller_state", "seller_city"]], on="seller_id", how="left", validate="many_to_one")
assert len(it) == n_it

it["categoria_original"] = it["product_category_name"].fillna("sem_categoria")
it["categoria_sem_traducao"] = it["product_category_name"].notna() & it["product_category_name_english"].isna()
it["categoria_en"] = it["product_category_name_english"].fillna(it["categoria_original"])
it["seller_regiao"] = it["seller_state"].map(U.UF_REGIAO)
registrar("itens_no_recorte", n_it)
registrar("itens_sem_categoria", int((it["categoria_original"] == "sem_categoria").sum()), "mantidos como 'sem_categoria'")
registrar("itens_categoria_sem_traducao", int(it["categoria_sem_traducao"].sum()), "nome original mantido")
registrar("itens_frete_zero", int((it["freight_value"] == 0).sum()), "valor válido (frete grátis), mantido")

pedidos_sem_itens = int((~ped["order_id"].isin(it["order_id"])).sum())
registrar("pedidos_recorte_sem_itens", pedidos_sem_itens)
print("Itens:", n_it, "| pedidos sem itens:", pedidos_sem_itens)

Itens: 85267 | pedidos sem itens: 0


## 5. Agregação de itens para o nível pedido
Para o KPI3 geral, o prazo de postagem do pedido é a **menor** `shipping_limit_date` entre os itens (regra da proposta). Soma de preço/frete é guardada só para conferência; o KPI2 é sempre calculado na tabela de itens.

In [6]:
agg_it = it.groupby("order_id").agg(
    qtd_itens=("order_item_id", "size"),
    qtd_sellers=("seller_id", "nunique"),
    prazo_postagem_limite=("shipping_limit_date", "min"),
    soma_preco=("price", "sum"),
    soma_frete=("freight_value", "sum"),
).reset_index()
ped = ped.merge(agg_it, on="order_id", how="left", validate="one_to_one")
assert len(ped) == n0 and ped["order_id"].is_unique

## 6. Variáveis derivadas
- **Postagem no prazo**: `order_delivered_carrier_date <= prazo_postagem_limite` (exatamente no limite = no prazo). Elegível só se as duas datas existirem.
- **Entrega atrasada**: `order_delivered_customer_date` (dia) > `order_estimated_delivery_date` (dia). A data estimada vem sem hora (00:00), por isso comparamos por **dia**: entregar no próprio dia estimado conta como no prazo.
- **Dias de atraso da entrega** = dia entregue − dia estimado (positivo = atraso; negativo = antecedência).
- **Intervalos** (em dias, com fração) só quando as duas datas existem; nada é imputado.

In [7]:
ped["postagem_tem_data"] = ped["order_delivered_carrier_date"].notna()
ped["postagem_tem_prazo"] = ped["prazo_postagem_limite"].notna()
ped["postagem_elegivel"] = ped["postagem_tem_data"] & ped["postagem_tem_prazo"]
ped["postagem_no_prazo"] = (ped["order_delivered_carrier_date"] <= ped["prazo_postagem_limite"]).where(ped["postagem_elegivel"])
ped["postagem_no_prazo"] = ped["postagem_no_prazo"].astype("boolean")
ped["postagem_atrasada"] = (~ped["postagem_no_prazo"]).astype("boolean")
ped["horas_atraso_postagem"] = (ped["order_delivered_carrier_date"] - ped["prazo_postagem_limite"]).dt.total_seconds() / 3600

registrar("kpi3_excluido_sem_data_postagem", int((~ped["postagem_tem_data"]).sum()))
registrar("kpi3_excluido_sem_prazo_limite", int((~ped["postagem_tem_prazo"]).sum()), "pedido sem itens")
registrar("postagem_antes_da_compra_no_recorte",
          int((ped["order_delivered_carrier_date"] < ped["order_purchase_timestamp"]).sum()),
          "data incoerente; mantido (não altera o lado do prazo)")

entrega_ok = ped["order_delivered_customer_date"].notna() & ped["order_estimated_delivery_date"].notna()
dia_ent = ped["order_delivered_customer_date"].dt.normalize()
dia_est = ped["order_estimated_delivery_date"].dt.normalize()
ped["dias_atraso_entrega"] = (dia_ent - dia_est).dt.days.where(entrega_ok)
ped["entrega_atrasada"] = (ped["dias_atraso_entrega"] > 0).where(entrega_ok).astype("boolean")
registrar("atraso_entrega_excluido_sem_data_entrega", int((~entrega_ok).sum()))

def dias(a, b):
    return (ped[b] - ped[a]).dt.total_seconds() / 86400
ped["dias_compra_aprovacao"] = dias("order_purchase_timestamp", "order_approved_at")
ped["dias_aprovacao_postagem"] = dias("order_approved_at", "order_delivered_carrier_date")
ped["dias_postagem_entrega"] = dias("order_delivered_carrier_date", "order_delivered_customer_date")
ped["dias_compra_entrega"] = dias("order_purchase_timestamp", "order_delivered_customer_date")

ped["situacao_postagem"] = ped["postagem_no_prazo"].map({True: "postagem no prazo", False: "postagem atrasada"}).fillna("sem dado")
ped["situacao_entrega"] = ped["entrega_atrasada"].map({True: "entrega atrasada", False: "entrega no prazo"}).fillna("sem dado")
ped[["postagem_elegivel", "postagem_no_prazo", "entrega_atrasada"]].describe()

,postagem_elegivel,postagem_no_prazo,entrega_atrasada
count,74781,74780,74774
unique,2,2,2
top,True,True,False
freq,74780,68374,69927


## 7. Tabela de itens final
Recebe período, mês, UF/região do cliente e as flags do pedido (útil para cortes de frete).

In [8]:
cols_ped = ["order_id", "ano_original", "periodo_comparativo", "mes", "ano_mes_original", "customer_state",
            "regiao_cliente", "avaliacao_ruim", "tem_avaliacao", "entrega_atrasada", "postagem_atrasada"]
itens = it.merge(ped[cols_ped], on="order_id", how="inner", validate="many_to_one")
assert len(itens) == n_it
assert not itens.duplicated(["order_id", "order_item_id"]).any()

## 8. Tabela pedido–seller
Para cada `order_id` + `seller_id`: menor `shipping_limit_date` dos itens daquele seller, comparada com a data de postagem do pedido (o dataset tem **uma** data de postagem por pedido). Um pedido com 2 sellers aparece 2 vezes aqui — **não** somar essa tabela como pedidos únicos.

In [9]:
ps = it.groupby(["order_id", "seller_id"]).agg(
    seller_state=("seller_state", "first"),
    seller_regiao=("seller_regiao", "first"),
    qtd_itens=("order_item_id", "size"),
    prazo_postagem_limite_seller=("shipping_limit_date", "min"),
    soma_preco=("price", "sum"),
    soma_frete=("freight_value", "sum"),
).reset_index()
ps = ps.merge(ped[["order_id", "ano_original", "periodo_comparativo", "mes", "customer_state", "regiao_cliente",
                   "order_delivered_carrier_date", "avaliacao_ruim", "tem_avaliacao", "entrega_atrasada"]],
              on="order_id", how="left", validate="many_to_one")
ps["postagem_elegivel"] = ps["order_delivered_carrier_date"].notna() & ps["prazo_postagem_limite_seller"].notna()
ps["postagem_no_prazo"] = (ps["order_delivered_carrier_date"] <= ps["prazo_postagem_limite_seller"]).where(ps["postagem_elegivel"]).astype("boolean")
ps["postagem_atrasada"] = (~ps["postagem_no_prazo"]).astype("boolean")
# semana (segunda-feira) do prazo limite: base para o ranking semanal
ps["semana_prazo_postagem"] = ps["prazo_postagem_limite_seller"].dt.to_period("W-SUN").dt.start_time
ps["granularidade"] = "pedido_seller"
assert not ps.duplicated(["order_id", "seller_id"]).any()
registrar("linhas_pedido_seller", len(ps), f"{ps['order_id'].nunique()} pedidos únicos")
print(len(ps), "linhas pedido-seller para", ps["order_id"].nunique(), "pedidos únicos")

75845 linhas pedido-seller para 74781 pedidos únicos


## 9. Tabela pedido–categoria
Um pedido pode ter itens de várias categorias. Para KPI1/KPI3 por categoria, o pedido é contado **uma vez em cada categoria distinta que contém** (soma das linhas > pedidos únicos; documentado).

In [10]:
pc = it[["order_id", "categoria_en", "categoria_original"]].drop_duplicates(["order_id", "categoria_en"])
pc = pc.merge(ped[["order_id", "ano_original", "periodo_comparativo", "tem_avaliacao", "avaliacao_ruim",
                   "postagem_elegivel", "postagem_no_prazo", "entrega_atrasada", "dias_atraso_entrega"]],
              on="order_id", how="left", validate="many_to_one")
registrar("pedidos_com_mais_de_uma_categoria", int((pc.groupby("order_id").size() > 1).sum()))
print(len(pc), "linhas pedido-categoria")

75408 linhas pedido-categoria


## 10. Salvando

In [11]:
ped["granularidade"] = "pedido"
itens["granularidade"] = "item"
pc["granularidade"] = "pedido_categoria"
for df, nome in [(ped, "pedidos"), (itens, "itens"), (ps, "pedido_seller"), (pc, "pedido_categoria")]:
    U.save_processed(df, nome)
    print(f"{nome:18s} {df.shape}")
U.save_processed(pd.DataFrame(descartes), "qualidade_02")
pd.DataFrame(descartes)[["verificacao", "valor", "detalhe"]]

pedidos            (74781, 41)


itens              (85267, 26)
pedido_seller      (75845, 22)


pedido_categoria   (75408, 12)


,verificacao,valor,detalhe
0,pedidos_base_total,99441,
1,descarte_status_diferente_de_delivered,2963,
2,descarte_delivered_fora_do_periodo,21697,"delivered, mas fora de jan-ago/2017 e jan-ago/..."
3,pedidos_no_recorte_2017,21998,referência da proposta: 21998
4,pedidos_no_recorte_2018,52783,referência da proposta: 52783
5,pedidos_recorte_com_multiplas_avaliacoes,391,mantida a mais recente
6,pedidos_recorte_sem_avaliacao,476,fora do denominador do KPI1
7,itens_no_recorte,85267,
8,itens_sem_categoria,1139,mantidos como 'sem_categoria'
9,itens_categoria_sem_traducao,22,nome original mantido
